# 🔍 Vector Stores & Retrievers — Hands-On Examples

> **Module 03 | Submodule 06 | Practical Notebook**
>
> Sections:
> 1. Embeddings — generate and compare vectors
> 2. Cosine similarity from scratch
> 3. Chroma — in-memory and persistent
> 4. FAISS — fast local vector search
> 5. Similarity search vs MMR comparison
> 6. Score threshold retrieval
> 7. EnsembleRetriever (hybrid BM25 + semantic)
> 8. MultiQueryRetriever
> 9. Complete RAG chain with retriever

In [ ]:
!pip install langchain langchain-openai langchain-community langchain-chroma \
             faiss-cpu chromadb rank_bm25 python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "module-03-vector-stores"

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm        = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Ready!")

In [ ]:
# Sample corpus of documents
docs = [
    Document(page_content="LangChain is a Python framework for building LLM-powered applications.",           metadata={"source": "intro",     "topic": "langchain",  "level": "beginner"}),
    Document(page_content="LangChain provides tools for prompting, chaining, and memory management.",        metadata={"source": "intro",     "topic": "langchain",  "level": "beginner"}),
    Document(page_content="LangGraph extends LangChain with stateful, graph-based agent orchestration.",     metadata={"source": "intro",     "topic": "langgraph",  "level": "intermediate"}),
    Document(page_content="In LangGraph, nodes are functions and edges define control flow between them.",    metadata={"source": "langgraph",  "topic": "langgraph",  "level": "intermediate"}),
    Document(page_content="LangGraph supports human-in-the-loop patterns for approvals and interventions.",  metadata={"source": "langgraph",  "topic": "langgraph",  "level": "advanced"}),
    Document(page_content="LangSmith provides tracing, monitoring, and evaluation for LangChain apps.",     metadata={"source": "ecosystem",  "topic": "langsmith",  "level": "intermediate"}),
    Document(page_content="FAISS is a library by Meta AI for efficient similarity search at scale.",          metadata={"source": "tools",      "topic": "faiss",      "level": "intermediate"}),
    Document(page_content="ChromaDB is an open-source vector database with rich metadata filtering.",         metadata={"source": "tools",      "topic": "chroma",     "level": "beginner"}),
    Document(page_content="LCEL (LangChain Expression Language) uses the pipe | operator to compose chains.", metadata={"source": "concepts",   "topic": "lcel",       "level": "beginner"}),
    Document(page_content="Retrieval-Augmented Generation grounds LLM answers in real retrieved documents.",  metadata={"source": "concepts",   "topic": "rag",        "level": "intermediate"}),
    Document(page_content="RAG pipelines: load documents → chunk → embed → store → retrieve → generate.",    metadata={"source": "concepts",   "topic": "rag",        "level": "beginner"}),
    Document(page_content="MMR (Maximal Marginal Relevance) retrieves relevant yet diverse documents.",      metadata={"source": "concepts",   "topic": "retrieval",  "level": "intermediate"}),
]
print(f"Corpus: {len(docs)} documents ready")

---
## 1️⃣ Embeddings — Generate and Inspect

In [ ]:
# Embed a single query
query_vec = embeddings.embed_query("What is LangChain?")
print(f"Query vector dimensions: {len(query_vec)}")
print(f"First 10 values: {[round(v, 4) for v in query_vec[:10]]}")
print(f"Min: {min(query_vec):.4f} | Max: {max(query_vec):.4f}")

In [ ]:
# Batch embed multiple documents
sample_texts = [doc.page_content for doc in docs[:4]]
doc_vecs = embeddings.embed_documents(sample_texts)

print(f"Documents embedded: {len(doc_vecs)}")
print(f"Each vector dimension: {len(doc_vecs[0])}")

---
## 2️⃣ Cosine Similarity from Scratch

In [ ]:
import numpy as np

def cosine_similarity(v1, v2):
    a, b = np.array(v1), np.array(v2)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Compare semantic similarity between sentences
test_sentences = [
    "LangChain is a Python framework for building AI applications",   # Similar to query
    "Python is a general-purpose programming language",               # Partial overlap
    "The Eiffel Tower is located in Paris, France",                   # Unrelated
    "LangGraph adds stateful agents to LangChain",                    # Related topic
]

query     = "What is LangChain used for?"
query_vec = embeddings.embed_query(query)
text_vecs = embeddings.embed_documents(test_sentences)

print(f"Query: '{query}'\n")
print(f"{'Score':>7}  {'Bar':20}  Text")
print("-" * 70)
for text, vec in sorted(
    zip(test_sentences, text_vecs),
    key=lambda x: cosine_similarity(query_vec, x[1]),
    reverse=True
):
    score = cosine_similarity(query_vec, vec)
    bar   = "█" * int(score * 25)
    print(f"  {score:.3f}  {bar:<20}  {text[:50]}")

---
## 3️⃣ Chroma Vector Store

In [ ]:
from langchain_chroma import Chroma

# Create in-memory Chroma store
chroma_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="module-06-demo"
)

print(f"Documents indexed: {chroma_store._collection.count()}")

In [ ]:
# Similarity search
results = chroma_store.similarity_search("How do I build stateful agents?", k=3)

print("Top 3 results for 'stateful agents':")
for i, doc in enumerate(results, 1):
    print(f"\n{i}. [{doc.metadata['topic']}] {doc.page_content}")

In [ ]:
# With relevance scores
results_scored = chroma_store.similarity_search_with_relevance_scores(
    "monitoring LangChain applications", k=4
)
print(f"{'Score':>7}  {'Topic':>12}  Content")
print("-" * 65)
for doc, score in results_scored:
    bar = "█" * int(score * 15)
    print(f"  {score:.3f}  {doc.metadata['topic']:>12}  {doc.page_content[:50]}")

In [ ]:
# Metadata filtering
results_filtered = chroma_store.similarity_search(
    query="building applications",
    k=3,
    filter={"level": "beginner"}   # Only beginner-level docs
)

print("Results filtered to 'beginner' level:")
for doc in results_filtered:
    print(f"  [{doc.metadata['level']}] {doc.page_content[:70]}")

In [ ]:
# Persistent Chroma
persist_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./chroma_persist_demo",
    collection_name="persistent-demo"
)
print("Saved to disk: ./chroma_persist_demo")

# Reload
reloaded = Chroma(
    persist_directory="./chroma_persist_demo",
    embedding_function=embeddings,
    collection_name="persistent-demo"
)
print(f"Reloaded: {reloaded._collection.count()} documents")

---
## 4️⃣ FAISS Vector Store

In [ ]:
from langchain_community.vectorstores import FAISS

faiss_store = FAISS.from_documents(docs, embedding=embeddings)
print(f"FAISS index created: {faiss_store.index.ntotal} vectors")

In [ ]:
results = faiss_store.similarity_search_with_score("vector search tools", k=3)
print("FAISS results (L2 distance — lower = closer):")
for doc, score in results:
    print(f"  Score: {score:.4f} | {doc.page_content[:70]}")

In [ ]:
# Save and reload FAISS
faiss_store.save_local("./faiss_demo_index")
print("Saved to ./faiss_demo_index")

reloaded_faiss = FAISS.load_local(
    "./faiss_demo_index",
    embeddings=embeddings,
    allow_dangerous_deserialization=True
)
print(f"Reloaded: {reloaded_faiss.index.ntotal} vectors")

---
## 5️⃣ Similarity Search vs MMR

In [ ]:
query = "LangChain framework"

# Standard similarity
sim_results = faiss_store.similarity_search(query, k=4)

# MMR — more diverse
mmr_results = faiss_store.max_marginal_relevance_search(
    query, k=4, fetch_k=10, lambda_mult=0.5
)

print(f"Query: '{query}'\n")
print("=== Similarity Search ===")
for i, doc in enumerate(sim_results, 1):
    print(f"  {i}. [{doc.metadata['topic']:12}] {doc.page_content[:65]}")

print("\n=== MMR (diverse) ===")
for i, doc in enumerate(mmr_results, 1):
    print(f"  {i}. [{doc.metadata['topic']:12}] {doc.page_content[:65]}")

# Note how MMR covers different topics, while similarity may repeat

---
## 6️⃣ Score Threshold Retriever

In [ ]:
# Only return high-confidence matches
threshold_retriever = chroma_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.75, "k": 5}
)

for query in ["monitoring LangChain", "what is croissant recipe"]:
    results = threshold_retriever.invoke(query)
    print(f"Query: '{query}' → {len(results)} confident match(es)")
    for doc in results:
        print(f"  ✅ {doc.page_content[:70]}")

---
## 7️⃣ EnsembleRetriever — Hybrid Search

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

# Keyword retriever (BM25)
bm25 = BM25Retriever.from_documents(docs)
bm25.k = 3

# Semantic retriever (FAISS)
semantic = faiss_store.as_retriever(search_kwargs={"k": 3})

# Hybrid: 40% keyword + 60% semantic
ensemble = EnsembleRetriever(
    retrievers=[bm25, semantic],
    weights=[0.4, 0.6]
)

test_queries = [
    "FAISS similarity search",     # Keyword match should help
    "how do I monitor my app",      # Semantic should help
    "stateful graph orchestration", # Both should find LangGraph docs
]

for q in test_queries:
    results = ensemble.invoke(q)
    print(f"\nQuery: '{q}'")
    for doc in results[:2]:
        print(f"  → [{doc.metadata['topic']:12}] {doc.page_content[:60]}")

---
## 8️⃣ MultiQueryRetriever

In [ ]:
import logging
from langchain.retrievers.multi_query import MultiQueryRetriever

# Enable logging to see generated sub-queries
logging.basicConfig(level=logging.WARNING)
mq_logger = logging.getLogger("langchain.retrievers.multi_query")
mq_logger.setLevel(logging.INFO)

mq_retriever = MultiQueryRetriever.from_llm(
    retriever=chroma_store.as_retriever(search_kwargs={"k": 2}),
    llm=llm
)

query = "How do LangChain agents decide what to do next?"
print(f"Query: '{query}'")
print("(watch INFO logs for generated sub-queries)\n")

results = mq_retriever.invoke(query)
print(f"\nUnique documents retrieved: {len(results)}")
for doc in results:
    print(f"  - [{doc.metadata['topic']:12}] {doc.page_content[:70]}")

---
## 9️⃣ Complete RAG Chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Use ensemble retriever for best results
retriever = ensemble  # BM25 + FAISS hybrid

def format_docs(docs):
    texts = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "unknown")
        texts.append(f"[{i}] ({source}) {doc.page_content}")
    return "\n".join(texts)

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful LangChain expert.
Answer ONLY using the provided context. If unsure, say so.

Context:
{context}"""),
    ("human", "{question}")
])

# Full RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Test questions
questions = [
    "What is LangGraph and how does it extend LangChain?",
    "What is the difference between FAISS and Chroma?",
    "What does LangSmith do?",
]

for q in questions:
    print(f"\n🤔 Q: {q}")
    answer = rag_chain.invoke(q)
    print(f"💡 A: {answer}")
    print("-" * 60)

---
## ✅ Summary

| Component | Purpose | Key Method |
|---|---|---|
| `OpenAIEmbeddings` | Text → vector | `.embed_query()`, `.embed_documents()` |
| `Chroma` | Persistent vector store with filtering | `.from_documents()`, `.similarity_search()` |
| `FAISS` | Fast in-memory vector store | `.from_documents()`, `.save_local()` |
| `similarity_search` | Find k closest docs | `vectorstore.similarity_search(query, k)` |
| `MMR` | Diverse + relevant retrieval | `max_marginal_relevance_search(...)` |
| `EnsembleRetriever` | Hybrid BM25 + semantic | Best retrieval quality |
| `MultiQueryRetriever` | Query expansion | Better recall for complex questions |

**Next**: [07 — Memory & Conversation History](../07_memory_and_conversation_history/theory.md)